In [1]:
import os
from stlf_torch_kit import  DataLoadeing
import torch
import pandas as pd
import numpy as np
import pickle, time
from stlf_torch_kit import univariate_multi_step
from stlf_torch_kit import SaveBestModel, PlotLossCurves, LoadModel, train, TestModel, BatchGenerator, results
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary


In [2]:
from platform import python_version
print(python_version())

3.7.0


# Data Loading

In [2]:
path_dataset =r'G:\HYBRID_CODE\CNN_LSTM\DATASETS' #Edit
path_tr = os.path.join(path_dataset, 'AEP_train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.values
path_v = os.path.join(path_dataset, 'AEP_validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.values 
path_te = os.path.join(path_dataset, 'AEP_test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

((84907, 21), (24259, 21), (12130, 21))

In [3]:
time_steps=24 #look back or sequence length, lag, window size #Edit
target_len = 1 #how much steps do you wana forecast #Edit
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=target_len)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=target_len)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=target_len)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.8995907306671143 sec


In [4]:
test_X.shape

(12105, 24, 21)

# Proposed Model

#### LSTM

In [5]:
# major edit
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.device = ('cuda' if torch.cuda.is_available() else 'cpu')
        self.lstm = nn.LSTM(1, 20, 1, batch_first=True).to(self.device) # num-features, node/units, num-layers
        self.fc = nn.Linear(20, 1).to(self.device) #(in,out)

    def forward(self, x):
        out, _ = self.lstm(x) # _=(h,c)
        out = self.fc(out[:, -1, :]) 
        return out

In [6]:
model = LSTMModel()
print(summary(model, input_size=(64, 24, 1)))

Layer (type:depth-idx)                   Output Shape              Param #
LSTMModel                                [64, 1]                   --
├─LSTM: 1-1                              [64, 24, 20]              1,840
├─Linear: 1-2                            [64, 1]                   21
Total params: 1,861
Trainable params: 1,861
Non-trainable params: 0
Total mult-adds (M): 2.83
Input size (MB): 0.01
Forward/backward pass size (MB): 0.25
Params size (MB): 0.01
Estimated Total Size (MB): 0.26


#### CNN
<table style="border: 1px solid black; border-collapse: collapse; width: 100%;">
    <thead>
        <tr style="background-color: #f2f2f2;">
            <th style="border: 1px solid black; padding: 8px;">Layer</th>
            <th style="border: 1px solid black; padding: 8px;">Expected Input Shape</th>
            <th style="border: 1px solid black; padding: 8px;">Common Adjustments Needed</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="border: 1px solid black; padding: 8px;">Conv1d</td>
            <td style="border: 1px solid black; padding: 8px;">(batch_size, channels, sequence_length)</td>
            <td style="border: 1px solid black; padding: 8px;">Permute input from (batch, seq, feat) to (batch, feat, seq)</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 8px;">LSTM</td>
            <td style="border: 1px solid black; padding: 8px;">(batch_size, sequence_length, features)</td>
            <td style="border: 1px solid black; padding: 8px;">No permutation if batch_first=True</td>
        </tr>
    </tbody>
</table>



In [7]:
class CNN1DModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.device = ('cuda' if torch.cuda.is_available() else 'cpu')
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3).to(self.device)
        self.fc = nn.Linear(32*22, 1).to(self.device) #32 channels ao 22 features dy chy kernel size 3 na bad kam shwal

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = torch.relu(self.conv1(x))  
        x = x.reshape(x.shape[0], -1)
        out = self.fc(x) 
        return out


In [8]:
model = CNN1DModel()
print(summary(model, input_size=(64, 24, 1)))
# summary(model, input_size=(64, 24, 1), col_names=["input_size", "output_size", "num_params", "kernel_size"])

Layer (type:depth-idx)                   Output Shape              Param #
CNN1DModel                               [64, 1]                   --
├─Conv1d: 1-1                            [64, 32, 22]              128
├─Linear: 1-2                            [64, 1]                   705
Total params: 833
Trainable params: 833
Non-trainable params: 0
Total mult-adds (M): 0.23
Input size (MB): 0.01
Forward/backward pass size (MB): 0.36
Params size (MB): 0.00
Estimated Total Size (MB): 0.37


In [5]:
class CNNLSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.device = ('cuda' if torch.cuda.is_available() else 'cpu')
        self.conv1 = nn.Conv1d(in_channels=21, out_channels=64, kernel_size=3).to(self.device)
       
        self.lstm = nn.LSTM(64, 50, 3, batch_first=True).to(self.device)
        self.fc = nn.Linear(50, 20).to(self.device) 
        self.fc1 = nn.Linear(20, 1).to(self.device) 


    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = torch.relu(self.conv1(x))
        
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x) # _=(h,c)
        out = self.fc(out[:, -1, :])
        out = self.fc1(out)
        return out


In [6]:
model = CNNLSTMModel()
print(summary(model, input_size=(64, 24, 21)))

Layer (type:depth-idx)                   Output Shape              Param #
CNNLSTMModel                             [64, 1]                   --
├─Conv1d: 1-1                            [64, 64, 22]              4,096
├─LSTM: 1-2                              [64, 22, 50]              64,000
├─Linear: 1-3                            [64, 20]                  1,020
├─Linear: 1-4                            [64, 1]                   21
Total params: 69,137
Trainable params: 69,137
Non-trainable params: 0
Total mult-adds (M): 95.95
Input size (MB): 0.13
Forward/backward pass size (MB): 1.29
Params size (MB): 0.28
Estimated Total Size (MB): 1.70


#### Mostly Cited

In [60]:
class MOSTLY_CITED(nn.Module):
    def __init__(self):
        super().__init__()
        self.device = ('cuda' if torch.cuda.is_available() else 'cpu')
        self.lstm = nn.LSTM(1, 20, 2, batch_first=True).to(self.device)
        self.fc = nn.Linear(20, 1).to(self.device) 

    def forward(self, x):
        out, _ = self.lstm(x) # _=(h,c)
        out = torch.sigmoid(self.fc(out[:, -1, :]) )
        return out


In [101]:
'cuda' if torch.cuda.is_available() else 'cpu'

'cuda'

# instances

In [7]:
model

CNNLSTMModel(
  (conv1): Conv1d(21, 64, kernel_size=(3,), stride=(1,))
  (lstm): LSTM(64, 50, num_layers=3, batch_first=True)
  (fc): Linear(in_features=50, out_features=20, bias=True)
  (fc1): Linear(in_features=20, out_features=1, bias=True)
)

In [8]:
model = CNNLSTMModel()#Edit
criterion = nn.MSELoss() #Edit, don't change

save_best_model = SaveBestModel()
Plot_Loss=PlotLossCurves()
load_model=LoadModel()

# Learning Rate & Optimizer

In [9]:
lr=0.001 # Edit
optimizer = torch.optim.Adam(model.parameters(), lr=lr) #Edit

# Check Device

In [10]:
def get_model_device(model):
    return next(model.parameters()).device
device = get_model_device(model)
print("Model is on device:", device)

Model is on device: cpu


# Training

#### Path & other Stuffs

In [11]:
start_epoch = 1
num_epochs = 10 #Edit
best_model_path=r'G:\HYBRID_CODE\CNN_LSTM\MODEL\6_CHK'+str('\\') #Edit
fig_path=r'G:\HYBRID_CODE\CNN_LSTM\MODEL\6_CHK' #Edit
train_data_loader, validation_data_loader, test_data_loader = DataLoadeing(train_X ,
                                                                           train_y, 
                                                                           validation_X, 
                                                                           validation_y, 
                                                                           test_X, 
                                                                           test_y, 
                                                                           batch_size=32) #Batch_Size Edit

#### Instances

In [12]:
criterion = nn.MSELoss() # Edit, for Now Don't  Change
save_best_model = SaveBestModel()
Plot_Loss=PlotLossCurves()
load_model=LoadModel()

#### Training Loop

In [118]:
start = time.time()
train(start_epoch,
      num_epochs,
      best_model_path,
      fig_path,
      model,criterion,optimizer,save_best_model,Plot_Loss,
      train_data_loader,
      validation_data_loader)
print('Time Consumed', time.time()-start, "sec")

Epoch [1/20], Step [2653/2653], Training Loss: 0.0097
Epoch [1/20], Step [758/758], Val Loss: 0.0107

Saving best model for epoch: 1
 at D:\chk5\1best_model.pth
Epoch [2/20], Step [2653/2653], Training Loss: 0.0046
Epoch [2/20], Step [758/758], Val Loss: 0.0063

Saving best model for epoch: 2
 at D:\chk5\2best_model.pth
Epoch [3/20], Step [2653/2653], Training Loss: 0.0009
Epoch [3/20], Step [758/758], Val Loss: 0.0012

Saving best model for epoch: 3
 at D:\chk5\3best_model.pth
Epoch [4/20], Step [2653/2653], Training Loss: 0.0004
Epoch [4/20], Step [758/758], Val Loss: 0.0009

Saving best model for epoch: 4
 at D:\chk5\4best_model.pth
Epoch [5/20], Step [2653/2653], Training Loss: 0.0003
Epoch [5/20], Step [758/758], Val Loss: 0.0006

Saving best model for epoch: 5
 at D:\chk5\5best_model.pth
Epoch [6/20], Step [2653/2653], Training Loss: 0.0003
Epoch [6/20], Step [758/758], Val Loss: 0.0005

Saving best model for epoch: 6
 at D:\chk5\6best_model.pth
Epoch [7/20], Step [2653/2653], Tr

KeyboardInterrupt: 

#### Results

In [19]:
import torch
torch.cuda.empty_cache()

In [119]:
load_model_path=r'D:\chk5\6best_model.pth' # Edit
test_model=TestModel()
start = time.time()
y_pred_scaled=test_model(model, test_X,load_model,load_model_path,lr)
print('Time Consumed', time.time()-start, "sec")
results(scaler, y_pred_scaled,test_y)

# MAPE, MAE, RMSE

New lr = 0.001
Time Consumed 0.307236909866333 sec
Mean Absolute Error (MAE): 235.95
Median Absolute Error (MedAE): 184.82
Mean Squared Error (MSE): 95055.62
Root Mean Squared Error (RMSE): 308.31
Mean Absolute Percentage Error (MAPE): 1.63 %
Median Absolute Percentage Error (MDAPE): 1.27 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


# Fine Tunning 1

In [120]:
num_epochs = 100
lr=0.0001
Batch_size = True #batch size maintained

#### Load Model for tuning

In [121]:
if Batch_size is True:
    model_, optimizer, start_epoch= load_model(model, model_path=load_model_path,
                               lr=lr,
                               train_X=train_X,
                               train_y=train_y,
                               validation_X=validation_X,
                               validation_y=validation_y,
                               test_X=test_X,
                               test_y=test_y)

else:
    model_, optimizer, start_epoch, train_data_loader, validation_data_loader, test_data_loader = load_model(model,model_path=load_model_path,
                                                                                             lr=lr,
                                                                                             Batch_Size=32,
                                                                                             train_X=train_X,
                                                                                             train_y=train_y,
                                                                                             validation_X=validation_X,
                                                                                             validation_y=validation_y,
                                                                                             test_X=test_X,
                                                                                             test_y=test_y)


New lr = 0.0001


In [122]:
model.load_state_dict(model_)
start = time.time()

train(start_epoch,
      num_epochs,
      best_model_path,
      fig_path,
      model,criterion,optimizer,save_best_model,Plot_Loss,
      train_data_loader,
      validation_data_loader,
     load_model_epoch=start_epoch)
print('Time Consumed', time.time()-start, "sec")

Epoch [7/106], Step [2653/2653], Training Loss: 0.0001
Epoch [7/106], Step [758/758], Val Loss: 0.0002

Saving best model for epoch: 7
 at D:\chk5\7best_model.pth
Epoch [8/106], Step [2653/2653], Training Loss: 0.0001
Epoch [8/106], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 8
 at D:\chk5\8best_model.pth
Epoch [9/106], Step [2653/2653], Training Loss: 0.0001
Epoch [9/106], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 9
 at D:\chk5\9best_model.pth
Epoch [10/106], Step [2653/2653], Training Loss: 0.0001
Epoch [10/106], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 10
 at D:\chk5\10best_model.pth
Epoch [11/106], Step [2653/2653], Training Loss: 0.0001
Epoch [11/106], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 11
 at D:\chk5\11best_model.pth
Epoch [12/106], Step [2653/2653], Training Loss: 0.0001
Epoch [12/106], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 12
 at D:\chk5\12best_model.pth
Epoch [13/

Epoch [58/106], Step [2653/2653], Training Loss: 0.0001
Epoch [58/106], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 58
 at D:\chk5\58best_model.pth
Epoch [59/106], Step [2653/2653], Training Loss: 0.0001
Epoch [59/106], Step [758/758], Val Loss: 0.0001
Epoch [60/106], Step [2653/2653], Training Loss: 0.0001
Epoch [60/106], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 60
 at D:\chk5\60best_model.pth
Epoch [61/106], Step [2653/2653], Training Loss: 0.0001
Epoch [61/106], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 61
 at D:\chk5\61best_model.pth
Epoch [62/106], Step [2653/2653], Training Loss: 0.0001
Epoch [62/106], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 62
 at D:\chk5\62best_model.pth
Epoch [63/106], Step [2653/2653], Training Loss: 0.0001
Epoch [63/106], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 63
 at D:\chk5\63best_model.pth
Epoch [64/106], Step [2653/2653], Training Loss: 0.0001
Epoc

In [124]:
load_model_path=r'D:\chk5\100best_model.pth'
start = time.time()
y_pred_scaled=test_model(model, test_X,load_model,load_model_path,lr)
print('Time Consumed', time.time()-start, "sec")
results(scaler, y_pred_scaled,test_y)

New lr = 0.0001
Time Consumed 0.3211486339569092 sec
Mean Absolute Error (MAE): 92.23
Median Absolute Error (MedAE): 73.08
Mean Squared Error (MSE): 15005.78
Root Mean Squared Error (RMSE): 122.5
Mean Absolute Percentage Error (MAPE): 0.63 %
Median Absolute Percentage Error (MDAPE): 0.51 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


# Fine Tunning 2

In [128]:
num_epochs = 20
lr = 0.00001
Batch_size=32
load_model_path=r'D:\chk5\100best_model.pth'

In [129]:
if Batch_size is True:
    model_, optimizer, start_epoch= load_model(model, model_path=load_model_path,
                               lr=lr,
                               train_X=train_X,
                               train_y=train_y,
                               validation_X=validation_X,
                               validation_y=validation_y,
                               test_X=test_X,
                               test_y=test_y)

else:
    model_, optimizer, start_epoch, train_data_loader, validation_data_loader, test_data_loader = load_model(model,model_path=load_model_path,
                                                                                             lr=lr,
                                                                                             Batch_Size=Batch_size,
                                                                                             train_X=train_X,
                                                                                             train_y=train_y,
                                                                                             validation_X=validation_X,
                                                                                             validation_y=validation_y,
                                                                                             test_X=test_X,
                                                                                             test_y=test_y)


New lr = 1e-05


In [130]:
model.load_state_dict(model_)
start = time.time()

train(start_epoch,
      num_epochs,
      best_model_path,
      fig_path,
      model,criterion,optimizer,save_best_model,Plot_Loss,
      train_data_loader,
      validation_data_loader,
     load_model_epoch=start_epoch)
print('Time Consumed', time.time()-start, "sec")

Epoch [101/120], Step [2653/2653], Training Loss: 0.0001
Epoch [101/120], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 101
 at D:\chk5\101best_model.pth
Epoch [102/120], Step [2653/2653], Training Loss: 0.0001
Epoch [102/120], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 102
 at D:\chk5\102best_model.pth
Epoch [103/120], Step [2653/2653], Training Loss: 0.0001
Epoch [103/120], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 103
 at D:\chk5\103best_model.pth
Epoch [104/120], Step [2653/2653], Training Loss: 0.0000
Epoch [104/120], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 104
 at D:\chk5\104best_model.pth
Epoch [105/120], Step [2653/2653], Training Loss: 0.0000
Epoch [105/120], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 105
 at D:\chk5\105best_model.pth
Epoch [106/120], Step [2653/2653], Training Loss: 0.0000
Epoch [106/120], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 106
 at 

In [131]:
load_model_path=r'D:\chk5\120best_model.pth'
test_model=TestModel()
start = time.time()
y_pred_scaled=test_model(model, test_X,load_model,load_model_path,lr)
print('Time Consumed', time.time()-start, "sec")
results(scaler, y_pred_scaled,test_y)

New lr = 1e-05
Time Consumed 0.1466076374053955 sec
Mean Absolute Error (MAE): 90.98
Median Absolute Error (MedAE): 71.16
Mean Squared Error (MSE): 14853.66
Root Mean Squared Error (RMSE): 121.88
Mean Absolute Percentage Error (MAPE): 0.62 %
Median Absolute Percentage Error (MDAPE): 0.5 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


In [132]:
num_epochs = 40
#lr = 0.0001
Batch_size=32
load_model_path=r'D:\chk5\120best_model.pth'

In [133]:
if Batch_size is True:
    model_, optimizer, start_epoch= load_model(model, model_path=load_model_path,
                               lr=lr,
                               train_X=train_X,
                               train_y=train_y,
                               validation_X=validation_X,
                               validation_y=validation_y,
                               test_X=test_X,
                               test_y=test_y)

else:
    model_, optimizer, start_epoch, train_data_loader, validation_data_loader, test_data_loader = load_model(model,model_path=load_model_path,
                                                                                             lr=lr,
                                                                                             Batch_Size=Batch_size,
                                                                                             train_X=train_X,
                                                                                             train_y=train_y,
                                                                                             validation_X=validation_X,
                                                                                             validation_y=validation_y,
                                                                                             test_X=test_X,
                                                                                             test_y=test_y)


New lr = 1e-05


In [134]:
model.load_state_dict(model_)
start = time.time()

train(start_epoch,
      num_epochs,
      best_model_path,
      fig_path,
      model,criterion,optimizer,save_best_model,Plot_Loss,
      train_data_loader,
      validation_data_loader,
     load_model_epoch=start_epoch)
print('Time Consumed', time.time()-start, "sec")

Epoch [121/160], Step [2653/2653], Training Loss: 0.0000
Epoch [121/160], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 121
 at D:\chk5\121best_model.pth
Epoch [122/160], Step [2653/2653], Training Loss: 0.0000
Epoch [122/160], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 122
 at D:\chk5\122best_model.pth
Epoch [123/160], Step [2653/2653], Training Loss: 0.0000
Epoch [123/160], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 123
 at D:\chk5\123best_model.pth
Epoch [124/160], Step [2653/2653], Training Loss: 0.0000
Epoch [124/160], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 124
 at D:\chk5\124best_model.pth
Epoch [125/160], Step [2653/2653], Training Loss: 0.0000
Epoch [125/160], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 125
 at D:\chk5\125best_model.pth
Epoch [126/160], Step [2653/2653], Training Loss: 0.0000
Epoch [126/160], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 126
 at 

In [135]:
load_model_path=r'D:\chk5\160best_model.pth'
test_model=TestModel()
start = time.time()
y_pred_scaled=test_model(model, test_X,load_model,load_model_path,lr)
print('Time Consumed', time.time()-start, "sec")
results(scaler, y_pred_scaled,test_y)

New lr = 1e-05
Time Consumed 0.1286466121673584 sec
Mean Absolute Error (MAE): 90.34
Median Absolute Error (MedAE): 70.28
Mean Squared Error (MSE): 14669.05
Root Mean Squared Error (RMSE): 121.12
Mean Absolute Percentage Error (MAPE): 0.62 %
Median Absolute Percentage Error (MDAPE): 0.49 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


In [139]:
num_epochs = 40
lr = 0.00001
Batch_size=32
load_model_path=r'D:\chk5\160best_model.pth'

In [140]:
if Batch_size is True:
    model_, optimizer, start_epoch= load_model(model, model_path=load_model_path,
                               lr=lr,
                               train_X=train_X,
                               train_y=train_y,
                               validation_X=validation_X,
                               validation_y=validation_y,
                               test_X=test_X,
                               test_y=test_y)

else:
    model_, optimizer, start_epoch, train_data_loader, validation_data_loader, test_data_loader = load_model(model,model_path=load_model_path,
                                                                                             lr=lr,
                                                                                             Batch_Size=Batch_size,
                                                                                             train_X=train_X,
                                                                                             train_y=train_y,
                                                                                             validation_X=validation_X,
                                                                                             validation_y=validation_y,
                                                                                             test_X=test_X,
                                                                                             test_y=test_y)


New lr = 1e-05


In [141]:
model.load_state_dict(model_)
start = time.time()

train(start_epoch,
      num_epochs,
      best_model_path,
      fig_path,
      model,criterion,optimizer,save_best_model,Plot_Loss,
      train_data_loader,
      validation_data_loader,
     load_model_epoch=start_epoch)
print('Time Consumed', time.time()-start, "sec")

Epoch [161/200], Step [2653/2653], Training Loss: 0.0000
Epoch [161/200], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 161
 at D:\chk5\161best_model.pth
Epoch [162/200], Step [2653/2653], Training Loss: 0.0000
Epoch [162/200], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 162
 at D:\chk5\162best_model.pth
Epoch [163/200], Step [2653/2653], Training Loss: 0.0000
Epoch [163/200], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 163
 at D:\chk5\163best_model.pth
Epoch [164/200], Step [2653/2653], Training Loss: 0.0000
Epoch [164/200], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 164
 at D:\chk5\164best_model.pth
Epoch [165/200], Step [2653/2653], Training Loss: 0.0000
Epoch [165/200], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 165
 at D:\chk5\165best_model.pth
Epoch [166/200], Step [2653/2653], Training Loss: 0.0000
Epoch [166/200], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 166
 at 

In [142]:
load_model_path=r'D:\chk5\200best_model.pth'
test_model=TestModel()
start = time.time()
y_pred_scaled=test_model(model, test_X,load_model,load_model_path,lr)
print('Time Consumed', time.time()-start, "sec")
results(scaler, y_pred_scaled,test_y)

New lr = 1e-05
Time Consumed 0.3027164936065674 sec
Mean Absolute Error (MAE): 89.99
Median Absolute Error (MedAE): 70.15
Mean Squared Error (MSE): 14559.41
Root Mean Squared Error (RMSE): 120.66
Mean Absolute Percentage Error (MAPE): 0.61 %
Median Absolute Percentage Error (MDAPE): 0.49 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


In [143]:
num_epochs = 60
#lr = 0.0001
Batch_size=32
load_model_path=r'D:\chk5\200best_model.pth'

In [144]:
if Batch_size is True:
    model_, optimizer, start_epoch= load_model(model, model_path=load_model_path,
                               lr=lr,
                               train_X=train_X,
                               train_y=train_y,
                               validation_X=validation_X,
                               validation_y=validation_y,
                               test_X=test_X,
                               test_y=test_y)

else:
    model_, optimizer, start_epoch, train_data_loader, validation_data_loader, test_data_loader = load_model(model,model_path=load_model_path,
                                                                                             lr=lr,
                                                                                             Batch_Size=Batch_size,
                                                                                             train_X=train_X,
                                                                                             train_y=train_y,
                                                                                             validation_X=validation_X,
                                                                                             validation_y=validation_y,
                                                                                             test_X=test_X,
                                                                                             test_y=test_y)


New lr = 1e-05


In [145]:
model.load_state_dict(model_)
start = time.time()

train(start_epoch,
      num_epochs,
      best_model_path,
      fig_path,
      model,criterion,optimizer,save_best_model,Plot_Loss,
      train_data_loader,
      validation_data_loader,
     load_model_epoch=start_epoch)
print('Time Consumed', time.time()-start, "sec")

Epoch [201/260], Step [2653/2653], Training Loss: 0.0000
Epoch [201/260], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 201
 at D:\chk5\201best_model.pth
Epoch [202/260], Step [2653/2653], Training Loss: 0.0000
Epoch [202/260], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 202
 at D:\chk5\202best_model.pth
Epoch [203/260], Step [2653/2653], Training Loss: 0.0000
Epoch [203/260], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 203
 at D:\chk5\203best_model.pth
Epoch [204/260], Step [2653/2653], Training Loss: 0.0000
Epoch [204/260], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 204
 at D:\chk5\204best_model.pth
Epoch [205/260], Step [2653/2653], Training Loss: 0.0000
Epoch [205/260], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 205
 at D:\chk5\205best_model.pth
Epoch [206/260], Step [2653/2653], Training Loss: 0.0000
Epoch [206/260], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 206
 at 

In [146]:
load_model_path=r'D:\chk5\260best_model.pth'
test_model=TestModel()
start = time.time()
y_pred_scaled=test_model(model, test_X,load_model,load_model_path,lr)
print('Time Consumed', time.time()-start, "sec")
results(scaler, y_pred_scaled,test_y)

New lr = 1e-05
Time Consumed 0.28228116035461426 sec
Mean Absolute Error (MAE): 89.62
Median Absolute Error (MedAE): 69.72
Mean Squared Error (MSE): 14440.51
Root Mean Squared Error (RMSE): 120.17
Mean Absolute Percentage Error (MAPE): 0.61 %
Median Absolute Percentage Error (MDAPE): 0.49 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


In [147]:
num_epochs = 60
#lr = 0.0001
Batch_size=32
load_model_path=r'D:\chk5\260best_model.pth'

In [148]:
if Batch_size is True:
    model_, optimizer, start_epoch= load_model(model, model_path=load_model_path,
                               lr=lr,
                               train_X=train_X,
                               train_y=train_y,
                               validation_X=validation_X,
                               validation_y=validation_y,
                               test_X=test_X,
                               test_y=test_y)

else:
    model_, optimizer, start_epoch, train_data_loader, validation_data_loader, test_data_loader = load_model(model,model_path=load_model_path,
                                                                                             lr=lr,
                                                                                             Batch_Size=Batch_size,
                                                                                             train_X=train_X,
                                                                                             train_y=train_y,
                                                                                             validation_X=validation_X,
                                                                                             validation_y=validation_y,
                                                                                             test_X=test_X,
                                                                                             test_y=test_y)


New lr = 1e-05


In [ ]:
model.load_state_dict(model_)
start = time.time()

train(start_epoch,
      num_epochs,
      best_model_path,
      fig_path,
      model,criterion,optimizer,save_best_model,Plot_Loss,
      train_data_loader,
      validation_data_loader,
     load_model_epoch=start_epoch)
print('Time Consumed', time.time()-start, "sec")

Epoch [261/320], Step [2653/2653], Training Loss: 0.0000
Epoch [261/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 261
 at D:\chk5\261best_model.pth
Epoch [262/320], Step [2653/2653], Training Loss: 0.0000
Epoch [262/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 262
 at D:\chk5\262best_model.pth
Epoch [263/320], Step [2653/2653], Training Loss: 0.0000
Epoch [263/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 263
 at D:\chk5\263best_model.pth
Epoch [264/320], Step [2653/2653], Training Loss: 0.0000
Epoch [264/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 264
 at D:\chk5\264best_model.pth
Epoch [265/320], Step [2653/2653], Training Loss: 0.0000
Epoch [265/320], Step [758/758], Val Loss: 0.0001
Epoch [266/320], Step [2653/2653], Training Loss: 0.0000
Epoch [266/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 266
 at D:\chk5\266best_model.pth
Epoch [267/320], Step [2653/2653], Tra

Epoch [310/320], Step [2653/2653], Training Loss: 0.0000
Epoch [310/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 310
 at D:\chk5\310best_model.pth
Epoch [311/320], Step [2653/2653], Training Loss: 0.0000
Epoch [311/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 311
 at D:\chk5\311best_model.pth
Epoch [312/320], Step [2653/2653], Training Loss: 0.0000
Epoch [312/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 312
 at D:\chk5\312best_model.pth
Epoch [313/320], Step [2653/2653], Training Loss: 0.0000
Epoch [313/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 313
 at D:\chk5\313best_model.pth
Epoch [314/320], Step [2653/2653], Training Loss: 0.0000
Epoch [314/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 314
 at D:\chk5\314best_model.pth
Epoch [315/320], Step [2653/2653], Training Loss: 0.0000
Epoch [315/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 315
 at 

In [12]:
lr = 0.00001

In [13]:
load_model_path=r'D:\chk5\318best_model.pth'
test_model=TestModel()
start = time.time()
y_pred_scaled=test_model(model, test_X,load_model,load_model_path,lr)
print('Time Consumed', time.time()-start, "sec")
results(scaler, y_pred_scaled,test_y)

New lr = 1e-05
Time Consumed 0.39956188201904297 sec
Mean Absolute Error (MAE): 89.36
Median Absolute Error (MedAE): 69.47
Mean Squared Error (MSE): 14358.5
Root Mean Squared Error (RMSE): 119.83
Mean Absolute Percentage Error (MAPE): 0.61 %
Median Absolute Percentage Error (MDAPE): 0.48 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


In [14]:
best_model_path=r'D:\chk5'+str('\\') #Edit
fig_path=r'D:\chk5'

In [15]:
num_epochs = 60
#lr = 0.0001
Batch_size=32
load_model_path=r'D:\chk5\260best_model.pth'

In [16]:
if Batch_size is True:
    model_, optimizer, start_epoch= load_model(model, model_path=load_model_path,
                               lr=lr,
                               train_X=train_X,
                               train_y=train_y,
                               validation_X=validation_X,
                               validation_y=validation_y,
                               test_X=test_X,
                               test_y=test_y)

else:
    model_, optimizer, start_epoch, train_data_loader, validation_data_loader, test_data_loader = load_model(model,model_path=load_model_path,
                                                                                             lr=lr,
                                                                                             Batch_Size=Batch_size,
                                                                                             train_X=train_X,
                                                                                             train_y=train_y,
                                                                                             validation_X=validation_X,
                                                                                             validation_y=validation_y,
                                                                                             test_X=test_X,
                                                                                             test_y=test_y)


New lr = 1e-05


In [17]:
model.load_state_dict(model_)
start = time.time()

train(start_epoch,
      num_epochs,
      best_model_path,
      fig_path,
      model,criterion,optimizer,save_best_model,Plot_Loss,
      train_data_loader,
      validation_data_loader,
     load_model_epoch=start_epoch)
print('Time Consumed', time.time()-start, "sec")

Epoch [261/320], Step [2653/2653], Training Loss: 0.0000
Epoch [261/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 261
 at D:\chk5\261best_model.pth
Epoch [262/320], Step [2653/2653], Training Loss: 0.0000
Epoch [262/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 262
 at D:\chk5\262best_model.pth
Epoch [263/320], Step [2653/2653], Training Loss: 0.0000
Epoch [263/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 263
 at D:\chk5\263best_model.pth
Epoch [264/320], Step [2653/2653], Training Loss: 0.0000
Epoch [264/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 264
 at D:\chk5\264best_model.pth
Epoch [265/320], Step [2653/2653], Training Loss: 0.0000
Epoch [265/320], Step [758/758], Val Loss: 0.0001
Epoch [266/320], Step [2653/2653], Training Loss: 0.0000
Epoch [266/320], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 266
 at D:\chk5\266best_model.pth
Epoch [267/320], Step [2653/2653], Tra

In [18]:
load_model_path=r'D:\chk5\320best_model.pth'
test_model=TestModel()
start = time.time()
y_pred_scaled=test_model(model, test_X,load_model,load_model_path,lr)
print('Time Consumed', time.time()-start, "sec")
results(scaler, y_pred_scaled,test_y)

New lr = 1e-05
Time Consumed 0.31932806968688965 sec
Mean Absolute Error (MAE): 89.35
Median Absolute Error (MedAE): 69.52
Mean Squared Error (MSE): 14356.16
Root Mean Squared Error (RMSE): 119.82
Mean Absolute Percentage Error (MAPE): 0.61 %
Median Absolute Percentage Error (MDAPE): 0.49 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


In [19]:
num_epochs = 60
#lr = 0.0001
Batch_size=32
load_model_path=r'D:\chk5\320best_model.pth'

In [20]:
if Batch_size is True:
    model_, optimizer, start_epoch= load_model(model, model_path=load_model_path,
                               lr=lr,
                               train_X=train_X,
                               train_y=train_y,
                               validation_X=validation_X,
                               validation_y=validation_y,
                               test_X=test_X,
                               test_y=test_y)

else:
    model_, optimizer, start_epoch, train_data_loader, validation_data_loader, test_data_loader = load_model(model,model_path=load_model_path,
                                                                                             lr=lr,
                                                                                             Batch_Size=Batch_size,
                                                                                             train_X=train_X,
                                                                                             train_y=train_y,
                                                                                             validation_X=validation_X,
                                                                                             validation_y=validation_y,
                                                                                             test_X=test_X,
                                                                                             test_y=test_y)


New lr = 1e-05


In [ ]:
model.load_state_dict(model_)
start = time.time()

train(start_epoch,
      num_epochs,
      best_model_path,
      fig_path,
      model,criterion,optimizer,save_best_model,Plot_Loss,
      train_data_loader,
      validation_data_loader,
     load_model_epoch=start_epoch)
print('Time Consumed', time.time()-start, "sec")

Epoch [321/380], Step [2653/2653], Training Loss: 0.0000
Epoch [321/380], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 321
 at D:\chk5\321best_model.pth
Epoch [322/380], Step [2653/2653], Training Loss: 0.0000
Epoch [322/380], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 322
 at D:\chk5\322best_model.pth
Epoch [323/380], Step [2653/2653], Training Loss: 0.0000
Epoch [323/380], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 323
 at D:\chk5\323best_model.pth
Epoch [324/380], Step [2653/2653], Training Loss: 0.0000
Epoch [324/380], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 324
 at D:\chk5\324best_model.pth
Epoch [325/380], Step [2653/2653], Training Loss: 0.0000
Epoch [325/380], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 325
 at D:\chk5\325best_model.pth
Epoch [326/380], Step [2653/2653], Training Loss: 0.0000
Epoch [326/380], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 326
 at 

Epoch [369/380], Step [2653/2653], Training Loss: 0.0000
Epoch [369/380], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 369
 at D:\chk5\369best_model.pth
Epoch [370/380], Step [2653/2653], Training Loss: 0.0000
Epoch [370/380], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 370
 at D:\chk5\370best_model.pth
Epoch [371/380], Step [2653/2653], Training Loss: 0.0000
Epoch [371/380], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 371
 at D:\chk5\371best_model.pth
Epoch [372/380], Step [2653/2653], Training Loss: 0.0000
Epoch [372/380], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 372
 at D:\chk5\372best_model.pth
Epoch [373/380], Step [2653/2653], Training Loss: 0.0000
Epoch [373/380], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 373
 at D:\chk5\373best_model.pth
Epoch [374/380], Step [2653/2653], Training Loss: 0.0000
Epoch [374/380], Step [758/758], Val Loss: 0.0001
Epoch [375/380], Step [2653/2653], Tra

In [10]:
lr = 0.00001

In [11]:
load_model_path=r'D:\chk5\376best_model.pth'
test_model=TestModel()
start = time.time()
y_pred_scaled=test_model(model, test_X,load_model,load_model_path,lr)
print('Time Consumed', time.time()-start, "sec")
results(scaler, y_pred_scaled,test_y)

New lr = 1e-05
Time Consumed 0.43485236167907715 sec
Mean Absolute Error (MAE): 89.15
Median Absolute Error (MedAE): 69.46
Mean Squared Error (MSE): 14294.5
Root Mean Squared Error (RMSE): 119.56
Mean Absolute Percentage Error (MAPE): 0.61 %
Median Absolute Percentage Error (MDAPE): 0.48 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


In [12]:
best_model_path=r'D:\chk5'+str('\\') #Edit
fig_path=r'D:\chk5'

In [17]:
num_epochs = 60
#lr = 0.0001
Batch_size=32
load_model_path=r'D:\chk5\436best_model.pth'

In [18]:
if Batch_size is True:
    model_, optimizer, start_epoch= load_model(model, model_path=load_model_path,
                               lr=lr,
                               train_X=train_X,
                               train_y=train_y,
                               validation_X=validation_X,
                               validation_y=validation_y,
                               test_X=test_X,
                               test_y=test_y)

else:
    model_, optimizer, start_epoch, train_data_loader, validation_data_loader, test_data_loader = load_model(model,model_path=load_model_path,
                                                                                             lr=lr,
                                                                                             Batch_Size=Batch_size,
                                                                                             train_X=train_X,
                                                                                             train_y=train_y,
                                                                                             validation_X=validation_X,
                                                                                             validation_y=validation_y,
                                                                                             test_X=test_X,
                                                                                             test_y=test_y)


New lr = 1e-05


In [ ]:
model.load_state_dict(model_)
start = time.time()

train(start_epoch,
      num_epochs,
      best_model_path,
      fig_path,
      model,criterion,optimizer,save_best_model,Plot_Loss,
      train_data_loader,
      validation_data_loader,
     load_model_epoch=start_epoch)
print('Time Consumed', time.time()-start, "sec")

Epoch [437/496], Step [2653/2653], Training Loss: 0.0000
Epoch [437/496], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 437
 at D:\chk5\437best_model.pth
Epoch [438/496], Step [2653/2653], Training Loss: 0.0000
Epoch [438/496], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 438
 at D:\chk5\438best_model.pth
Epoch [439/496], Step [2653/2653], Training Loss: 0.0000
Epoch [439/496], Step [758/758], Val Loss: 0.0001
Epoch [440/496], Step [2653/2653], Training Loss: 0.0000
Epoch [440/496], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 440
 at D:\chk5\440best_model.pth
Epoch [441/496], Step [2653/2653], Training Loss: 0.0000
Epoch [441/496], Step [758/758], Val Loss: 0.0001

Saving best model for epoch: 441
 at D:\chk5\441best_model.pth
Epoch [442/496], Step [2653/2653], Training Loss: 0.0000
Epoch [442/496], Step [758/758], Val Loss: 0.0001
Epoch [443/496], Step [2653/2653], Training Loss: 0.0000
Epoch [443/496], Step [758/758], Val Loss: 0.

In [16]:
load_model_path=r'D:\chk5\436best_model.pth'
test_model=TestModel()
start = time.time()
y_pred_scaled=test_model(model, test_X,load_model,load_model_path,lr)
print('Time Consumed', time.time()-start, "sec")
results(scaler, y_pred_scaled,test_y)

New lr = 1e-05
Time Consumed 0.2766132354736328 sec
Mean Absolute Error (MAE): 88.98
Median Absolute Error (MedAE): 69.03
Mean Squared Error (MSE): 14236.96
Root Mean Squared Error (RMSE): 119.32
Mean Absolute Percentage Error (MAPE): 0.61 %
Median Absolute Percentage Error (MDAPE): 0.48 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


In [27]:
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124


In [3]:
import torch

In [4]:
torch.cuda.is_available()

True

In [2]:
pip install torchinfo

  Using cached torchinfo-1.8.0-py3-none-any.whl (23 kB)
Note: you may need to restart the kernel to use updated packages.
